# CGRP migraine project, Part 1: Fetching the proteins

**Goal of the whole project:** use an MT-DTI-style transformer model to predict which small molecules bind the CGRP receptor and could block it.

**Lock and key:**
- The **key** is the CGRP peptide, made from the genes *CALCA* (alpha-CGRP) and *CALCB* (beta-CGRP).
- The **lock** is the CGRP receptor, built from two proteins: **CALCRL** (CLR) and **RAMP1**.
- Migraine drugs called gepants block the *lock*, so the lock is our drug target.

**This notebook:** download the four proteins from UniProt, read their sequences, find the real CGRP peptides, and save a clean table.

## Step 1: Fetch the proteins from UniProt

UniProt is a public database where every protein has a unique ID. We search by gene name instead of typing sequences by hand, to avoid copying errors.

In [ ]:
import requests
import pandas as pd

# gene name -> what it is (just a label for us)
genes = {
    "CALCA":  "CGRP peptide (alpha), the signal",
    "CALCB":  "CGRP peptide (beta), close relative",
    "CALCRL": "receptor half 1 (CLR)",
    "RAMP1":  "receptor half 2 (co-receptor)",
}

def fetch_protein(gene):
    """Ask UniProt for the reviewed human entry of a gene."""
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={
            "query": f"(gene_exact:{gene}) AND (organism_id:9606) AND (reviewed:true)",
            "format": "json",
            "size": 1,
        },
        timeout=30,
    )
    r.raise_for_status()
    hit = r.json()["results"][0]
    return {
        "gene": gene,
        "role": genes[gene],
        "uniprot_id": hit["primaryAccession"],
        "length_aa": hit["sequence"]["length"],
        "sequence": hit["sequence"]["value"],
    }

targets = pd.DataFrame([fetch_protein(g) for g in genes])
targets.to_csv("targets.csv", index=False)
print(targets[["gene", "role", "uniprot_id", "length_aa"]])

**Vocabulary**
- **gene**: a stretch of DNA with instructions for one protein.
- **UniProt ID**: the protein's unique database ID, like a library card number.
- **length_aa**: number of amino acids, the building blocks of a protein. 20 types, each written as a letter.

**Heads-up:** this search asked for only one result per gene. For CALCA, it returned P01258, which turns out to be the *thyroid* product (calcitonin), not the nerve product (CGRP). We fix that in Step 3.

## Step 2: Read the sequences

A protein sequence is the order of its amino acids, written as letters. This is what the model will read as input.

In [ ]:
from collections import Counter

targets = pd.read_csv("targets.csv")

for _, row in targets.iterrows():
    seq = row["sequence"]
    print(f"=== {row['gene']}  ({row['role']})  {len(seq)} amino acids ===")

    # print the sequence in lines of 60 letters so it's readable
    for i in range(0, len(seq), 60):
        print(f"{i+1:>4}  {seq[i:i+60]}")

    # which amino acids are most common?
    top = Counter(seq).most_common(5)
    print("Most common letters:", top)
    print()

**What the sequences show**
- Every sequence starts with **M** (the "start building" signal).
- CALCRL is rich in **L, I, V**: oily amino acids that sit in the fatty cell membrane. Long runs of them mark the stretches that cross the membrane (for example `LFYLTIIGHGLSIASLLISLGIFFYF` in CALCRL).
- RAMP1 has only one such stretch, at its end, so it crosses the membrane once. Most of it is outside the cell.
- CALCA and CALCB start almost identically, then diverge.
- The CGRP peptide is visible in CALCB but **not** in this CALCA entry.

## Step 3: Find the real CGRP pieces

Proteins are often made long and then cut. UniProt labels the pieces ("features"): signal piece, propeptides, active peptides. This helper lists them and cuts each out of the full sequence.

In [ ]:
def get_entry(acc):
    """Download the full UniProt record (as JSON) for one protein ID."""
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=30)
    r.raise_for_status()
    return r.json()

def show_pieces(acc):
    """List the labelled pieces (signal peptide, active peptides) inside a protein."""
    entry = get_entry(acc)
    seq = entry["sequence"]["value"]
    print(f"--- {acc}: {entry['genes'][0]['geneName']['value']}, {len(seq)} aa ---")
    for f in entry["features"]:
        if f["type"] in ("Signal", "Peptide", "Chain", "Propeptide"):
            start = f["location"]["start"]["value"]
            end = f["location"]["end"]["value"]
            piece = seq[start - 1:end]          # Python counts from 0, UniProt from 1
            print(f"{f['type']:<11} {start:>3}-{end:<3} ({len(piece)} aa)  {f['description']}")
            print(f"            {piece}")
    print()

show_pieces("P10092")   # CALCB
show_pieces("P01258")   # CALCA (the entry our search returned)

# CALCA also lists isoforms; check their lengths
for iso in ["P01258-1", "P01258-2", "P01258-3"]:
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{iso}.fasta", timeout=30)
    if r.status_code == 200:
        lines = r.text.strip().split("\n")
        print(lines[0])
        print(f"   length: {len(''.join(lines[1:]))} aa")
    else:
        print(iso, "not found")

**Findings**
- CALCB (P10092) contains **beta-CGRP**: a 37-aa peptide (positions 82-118).
- The CALCA entry P01258 contains **calcitonin** (32 aa) and **katacalcin**, the thyroid products. No CGRP.
- The two isoforms of P01258 are both 141 aa, so isoforms were not where alpha-CGRP lives.
- UniProt stores the nerve product of CALCA as its **own entry, P06881**. Next cell checks it.

In [ ]:
show_pieces("P06881")   # alpha-CGRP (nerve version of CALCA)

# Compare alpha and beta CGRP letter by letter
def get_peptide(acc, name_contains):
    entry = get_entry(acc)
    seq = entry["sequence"]["value"]
    for f in entry["features"]:
        if f["type"] == "Peptide" and name_contains in f["description"]:
            s, e = f["location"]["start"]["value"], f["location"]["end"]["value"]
            return seq[s - 1:e]

alpha = get_peptide("P06881", "gene-related peptide")
beta  = get_peptide("P10092", "gene-related peptide")
print("alpha:", alpha, len(alpha))
print("beta: ", beta, len(beta))
diffs = [(i + 1, a, b) for i, (a, b) in enumerate(zip(alpha, beta)) if a != b]
print(f"{len(diffs)} differences (position, alpha, beta):", diffs)

**Findings**
- **P06881 is alpha-CGRP**: 37 aa, "Calcitonin gene-related peptide 1".
- Alpha and beta CGRP differ at only **3 of 37 positions**: 3 (D/N), 22 (V/M), 25 (N/S). This is why one receptor responds to both.
- The two cysteines (C) at positions 2 and 7 link up and form a small ring that helps CGRP fit its receptor. A model that only reads letters cannot see this ring, which is a limitation to keep in mind.
- Lesson learned: always check that a database search returned the entry you meant. We asked for one result and got the wrong CALCA product.

## Step 4: Save a clean target table

Rebuild the table with the correct CGRP peptides and the two receptor proteins.

In [ ]:
old = pd.read_csv("targets.csv").set_index("gene")

clean = pd.DataFrame([
    {"name": "alpha-CGRP (key)", "uniprot_id": "P06881", "sequence": alpha},
    {"name": "beta-CGRP (key)",  "uniprot_id": "P10092", "sequence": beta},
    {"name": "CALCRL (lock, half 1)", "uniprot_id": old.loc["CALCRL", "uniprot_id"],
     "sequence": old.loc["CALCRL", "sequence"]},
    {"name": "RAMP1 (lock, half 2)",  "uniprot_id": old.loc["RAMP1", "uniprot_id"],
     "sequence": old.loc["RAMP1", "sequence"]},
])
clean["length_aa"] = clean.sequence.str.len()
clean.to_csv("targets_clean.csv", index=False)
print(clean[["name", "uniprot_id", "length_aa"]])

## Next: Part 2
Look at what the ChEMBL database contains for the CGRP receptor (how many compounds, how strongly they bind, which known drugs are included) to check the project is feasible before building any model.